<a href="https://colab.research.google.com/github/Stdcoders/Graph-RAG/blob/main/GraphRAG_L3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -r /content/Agentic_KAG_Workshop_DHS_2026/requirements.txt --quiet

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.7/263.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.0/358.0 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.7 MB/s eta 0:00:00
   ━━━━━

In [1]:
!git clone https://github.com/genaiconference/Agentic_KAG_Workshop_DHS_2026.git

Cloning into 'Agentic_KAG_Workshop_DHS_2026'...
remote: Enumerating objects: 403, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 403 (delta 125), reused 57 (delta 55), pack-reused 224 (from 2)
Receiving objects: 100% (403/403), 13.34 MiB | 25.53 MiB/s, done.
Resolving deltas: 100% (217/217), done.


In [6]:
import os

os.chdir('/content/Agentic_KAG_Workshop_DHS_2026/')

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    print("error reading env details")
    pass

# --- Neo4j Sandbox ---
NEO4J_URI      = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')

# --- OpenAI ---

print('NEO4J_URI :', NEO4J_URI)
# print('OPENAI key set:', bool(os.environ.get('OPENAI_API_KEY')))

NEO4J_URI : neo4j+s://313964e6.databases.neo4j.io


In [7]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Connected to Neo4j ✔')

Connected to Neo4j ✔


In [8]:
import os
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings.openai import BaseOpenAIEmbeddings

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

class NemotronEmbeddings(BaseOpenAIEmbeddings):
    """
    NVIDIA Nemotron embeddings via NIM's OpenAI-compatible /v1/embeddings endpoint.
    Nemotron-family embedding models require an `input_type` of "passage" (indexing)
    or "query" (retrieval) on every request, so we inject it automatically here.
    """
    def __init__(self, model="nvidia/nemotron-3-embed-1b", input_type="passage", **kwargs):
        self.input_type = input_type
        super().__init__(model=model, **kwargs)

    def _initialize_client(self, **kwargs):
        return self.openai.OpenAI(**kwargs)

    def embed_query(self, text, **kwargs):
        kwargs.setdefault("extra_body", {"input_type": self.input_type})
        return super().embed_query(text, **kwargs)

# Main LLM used by LLMEntityRelationExtractor
llm = OpenAILLM(
    model_name="nvidia/nemotron-3-super-120b-a12b",
    model_params={
        "response_format": {"type": "json_object"},
    },
    base_url=NVIDIA_BASE_URL,
    api_key=NVIDIA_API_KEY,
)

# One instance for indexing (KG builder embeds source text/passages)
passage_embedder = NemotronEmbeddings(
    model="nvidia/nemotron-3-embed-1b",
    base_url=NVIDIA_BASE_URL,
    api_key=NVIDIA_API_KEY,
    input_type="passage",
)

# A separate instance for the retriever at query time
query_embedder = NemotronEmbeddings(
    model="nvidia/nemotron-3-embed-1b",
    base_url=NVIDIA_BASE_URL,
    api_key=NVIDIA_API_KEY,
    input_type="query",
)

print('LLM + embedders ready ✔')

LLM + embedders ready ✔


In [9]:
from pathlib import Path
import pandas as pd
import math
import os
import json
import ast
from neo4j_graphrag.experimental.components.data_loader import DataLoader
from neo4j_graphrag.experimental.components.types import LoadedDocument

excel_path = "/content/Agentic_KAG_Workshop_DHS_2026/data/TMDB_IMDB_Movies_Dataset_filtered.xlsx"

DATA_PATH = Path(excel_path)

class MoviesDataLoader(DataLoader):
    """Custom DataLoader that reads a movies CSV and returns a `Document` object.

    The `text` field of the returned `Document` object is a `repr()` of a list of per-movie dicts. The splitter (next step) will `ast.literal_eval` it.
    """

    def __init__(self, **kwargs):
        pass

    async def run(self, path: Path) -> LoadedDocument:
        df = pd.read_excel(path)
        print(f"Initial DataFrame shape: {df.shape}")

        df = df.reset_index(drop=True)

        # Convert 'release_date' column to datetime and then to the desired string format.
        # NA values will be replaced to None after string formatting.
        if 'release_date' in df.columns:
            df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
            df['release_date'] = df['release_date'].dt.strftime('%Y-%m-%d').replace({pd.NA: None})

        # Convert all NaN values to None before converting to dictionary, as ast.literal_eval cannot parse 'nan'
        df = df.where(pd.notna(df), None)

        # Convert the DataFrame to a list of dictionaries.
        # This handles NaN/NaT values by converting them to None by default for all columns.
        records = df.to_dict(orient='records')

        print(f'Loaded {len(records)} movie rows from {path.name} (no filtering applied).')

        return LoadedDocument(
            text=repr(records),
            document_info={'path': str(path)},
        )


# Execute
loader = MoviesDataLoader()
page_data = await loader.run(DATA_PATH)

Initial DataFrame shape: (4869, 26)
Loaded 4869 movie rows from TMDB_IMDB_Movies_Dataset_filtered.xlsx (no filtering applied).


In [10]:
import ast
from neo4j_graphrag.experimental.components.text_splitters.base import TextSplitter
from neo4j_graphrag.experimental.components.types import TextChunks, TextChunk, LoadedDocument

def _row_to_movie_doc(rec: dict) -> str:
    """Build a multi-line, LLM-friendly text block for a single movie row."""
    lines = []

    # Define fields to display and their formatting
    display_fields = {
        'title': {'label': 'Title'},
        'release_date': {'label': 'Release date'},
        'runtime': {'label': 'Runtime (minutes)', 'formatter': lambda x: f"{x:.0f}"},
        'budget': {'label': 'Budget (USD)', 'formatter': lambda x: f"{x:,.0f}"},
        'revenue': {'label': 'Revenue (USD)', 'formatter': lambda x: f"{x:,.0f}"},
        'vote_average': {'label': 'Vote average', 'formatter': lambda x: f"{x:.1f}"},
        'genres': {'label': 'Genres'},
        'production_companies': {'label': 'Production companies'},
        'production_countries': {'label': 'Production countries'},
        'spoken_languages': {'label': 'Spoken languages'},
        'keywords': {'label': 'Keywords'},
        'directors': {'label': 'Director(s)'},
        'cast': {'label': 'Cast'},
        'tagline': {'label': 'Tagline'},
        'overview': {'label': 'Overview'},
    }

    for key, config in display_fields.items():
        value = rec.get(key)
        # Check for None, empty string, or empty collections (list, tuple, dict)
        if value is not None and value != '' and not (isinstance(value, (list, tuple, dict)) and not value):
            label = config['label']
            formatter = config.get('formatter')
            if formatter:
                # Apply formatter, handling potential errors if value is not numeric
                try:
                    formatted_value = formatter(value)
                except (TypeError, ValueError):
                    formatted_value = str(value) # Fallback to string conversion
            else:
                formatted_value = str(value)
            lines.append(f"{label}: {formatted_value}")

    return '\n'.join(lines)


class MoviesRowTextSplitter(TextSplitter):
    """Split the loader's Document into one TextChunk per movie row."""

    def __init__(self, dataset_name: str = 'TMDB+IMDb Movies'):
        self.dataset_name = dataset_name

    async def run(self, page_data: LoadedDocument) -> TextChunks:
        raw = page_data['text'] if isinstance(page_data, dict) else page_data.text
        records = ast.literal_eval(raw)

        chunks = []
        for i, rec in enumerate(records):
            chunks.append(TextChunk(
                index=i,
                text=_row_to_movie_doc(rec),
                metadata={
                    'title':        rec.get('title', ''),
                    'release_date': rec.get('release_date', ''),
                    'dataset':      self.dataset_name,
                },
            ))
        print(f'Built {len(chunks)} TextChunks (one per movie).')
        return TextChunks(chunks=chunks)


# Execute
splitter = MoviesRowTextSplitter()
text_chunks = await splitter.run(page_data)
print('*****First chunk preview:*******\n', text_chunks.chunks[0].text)

Built 4869 TextChunks (one per movie).
*****First chunk preview:*******
 Title: Eternals
Release date: 2021-11-03
Runtime (minutes): 156
Budget (USD): 200,000,000
Revenue (USD): 402,064,899
Vote average: 6.9
Genres: Science Fiction, Action, Adventure
Production companies: Marvel Studios
Production countries: Canada, United States of America
Spoken languages: Arabic, English, Hindi, Latin, Spanish
Keywords: superhero, supernatural, based on comic, alien, super power, aftercreditsstinger, marvel cinematic universe (mcu), sign languages, ancient, god-like
Director(s): Chloé Zhao
Cast: Gemma Chan, Richard Madden, Angelina Jolie, Salma Hayek Pinault, Kumail Nanjiani, Lia McHugh, Brian Tyree Henry, Lauren Ridloff, Barry Keoghan, Ma Dong-seok
Tagline: In the beginning...
Overview: The Eternals are a team of ancient aliens who have been living on Earth in secret for thousands of years. When an unexpected tragedy forces them out of the shadows, they are forced to reunite against mankind’s most 

In [11]:

from neo4j_graphrag.experimental.components.embedder import TextChunkEmbedder

text_chunk_embedder = TextChunkEmbedder(embedder=query_embedder)

In [12]:
from neo4j_graphrag.experimental.components.schema import (
    SchemaBuilder,
    NodeType,
    RelationshipType,
    GraphSchema,
)

schema_builder = SchemaBuilder()

# -------- Node Types --------
node_types = [
    NodeType(
        label='Movie',
        description='A motion picture / film with metadata such as title, runtime, budget, revenue, release date and ratings.',
        properties=[
            {'name': 'title',         'type': 'STRING',  'description': 'Title of the movie.'},
            {'name': 'release_date',  'type': 'DATE',    'description': 'Theatrical release date (YYYY-MM-DD).'},
            {'name': 'runtime_minutes',       'type': 'INTEGER', 'description': 'Runtime in minutes.'},
            {'name': 'budget_usd',        'type': 'INTEGER', 'description': 'Production budget in USD.'},
            {'name': 'revenue_usd',       'type': 'INTEGER', 'description': 'Worldwide box-office revenue in USD.'},
            {'name': 'vote_average',  'type': 'FLOAT',   'description': 'Average user rating (typically 0–10).'},
            {'name': 'overview',      'type': 'STRING',  'description': 'Plot synopsis.'},
            {'name': 'tagline',       'type': 'STRING',  'description': 'Marketing tagline of the movie.'},
        ],
        additional_properties=True,
    ),
    NodeType(
        label='Person',
        description='A real person involved in a movie as director, writer, actor or other crew.',
        properties=[
            {'name': 'name', 'type': 'STRING', 'description': 'Full name of the person.'},
        ],
        additional_properties=True,
    ),
    NodeType(
        label='Genre',
        description='A film genre such as Action, Drama, Comedy, Horror, Science Fiction.',
        properties=[
            {'name': 'name', 'type': 'STRING', 'description': 'Canonical genre name.'},
        ],
        additional_properties=True,
    ),
    NodeType(
        label='ProductionCompany',
        description='A company that produced or financed the movie.',
        properties=[
            {'name': 'name', 'type': 'STRING', 'description': 'Name of the production company.'},
        ],
        additional_properties=True,
    ),
    NodeType(
        label='Country',
        description='A country where the movie was produced.',
        properties=[
            {'name': 'name', 'type': 'STRING', 'description': 'Country name.'},
        ],
        additional_properties=True,
    ),
    NodeType(
        label='Language',
        description='A spoken language in the movie.',
        properties=[
            {'name': 'name', 'type': 'STRING', 'description': 'Language name (e.g. English, French).'},
        ],
        additional_properties=True,
    ),
    NodeType(
        label='Keyword',
        description='A descriptive tag / theme attached to the movie (e.g. "time travel", "heist").',
        properties=[
            {'name': 'name', 'type': 'STRING', 'description': 'Keyword text.'},
        ],
        additional_properties=True,
    ),
]

# -------- Relationship Types --------
relationship_types = [
    RelationshipType(label='HAS_GENRE',  description='Movie -[HAS_GENRE]-> Genre',                 properties=[], additional_properties=True),
    RelationshipType(label='PRODUCED_BY',description='Movie -[PRODUCED_BY]-> ProductionCompany',   properties=[], additional_properties=True),
    RelationshipType(label='PRODUCED_IN',description='Movie -[PRODUCED_IN]-> Country',             properties=[], additional_properties=True),
    RelationshipType(label='SPOKEN_IN',  description='Movie -[SPOKEN_IN]-> Language',              properties=[], additional_properties=True),
    RelationshipType(label='TAGGED_WITH',description='Movie -[TAGGED_WITH]-> Keyword',             properties=[], additional_properties=True),
    RelationshipType(label='DIRECTED_BY',description='Movie -[DIRECTED_BY]-> Person (director)',   properties=[], additional_properties=True),
    RelationshipType(label='WRITTEN_BY', description='Movie -[WRITTEN_BY]-> Person (writer)',      properties=[], additional_properties=True),
    RelationshipType(label='CAST_IN',    description='Person -[CAST_IN]-> Movie (actor appears in movie)', properties=[], additional_properties=True),
]

# -------- Patterns (allowed triplets) --------
patterns = [
    ('Movie',  'HAS_GENRE',   'Genre'),
    ('Movie',  'PRODUCED_BY', 'ProductionCompany'),
    ('Movie',  'PRODUCED_IN', 'Country'),
    ('Movie',  'SPOKEN_IN',   'Language'),
    ('Movie',  'TAGGED_WITH', 'Keyword'),
    ('Movie',  'DIRECTED_BY', 'Person'),
    ('Movie',  'WRITTEN_BY',  'Person'),
    ('Person', 'CAST_IN',     'Movie'),
]

# Build a manual schema using node, relationships and patterns
manual_schema = GraphSchema(
    node_types=node_types,
    relationship_types=relationship_types,
    patterns=patterns,
)

print('Schema ready:')
print('  node_types        :', [n.label for n in node_types])
print('  relationship_types:', [r.label for r in relationship_types])
print('  patterns          :', len(patterns))

Schema ready:
  node_types        : ['Movie', 'Person', 'Genre', 'ProductionCompany', 'Country', 'Language', 'Keyword']
  relationship_types: ['HAS_GENRE', 'PRODUCED_BY', 'PRODUCED_IN', 'SPOKEN_IN', 'TAGGED_WITH', 'DIRECTED_BY', 'WRITTEN_BY', 'CAST_IN']
  patterns          : 8


In [13]:
from neo4j_graphrag.experimental.components.entity_relation_extractor import (
    LLMEntityRelationExtractor,
    OnError,
)
import prompts

entity_extractor = LLMEntityRelationExtractor(
    llm=llm,
    on_error=OnError.IGNORE,
    create_lexical_graph=False,
    prompt_template=prompts.ENTITY_RELATION_EXTRACTOR_PROMPT,
)

In [14]:
from neo4j_graphrag.experimental.components.resolver import SinglePropertyExactMatchResolver

# The SinglePropertyExactMatchResolver performs entity resolution based on the specified 'resolve_property'.
# Here, 'resolver_title' will resolve entities based on their 'title' property.
resolver_title = SinglePropertyExactMatchResolver(driver=driver, resolve_property='title')
res_exactmatch = await resolver_title.run()

# And 'resolver_name' will resolve entities based on their 'name' property.
resolver_name  = SinglePropertyExactMatchResolver(driver=driver, resolve_property='name')
# res_exactmatch = await resolver_name.run()

In [15]:
from neo4j_graphrag.experimental.pipeline import Component, DataModel

class EmbedEntityResult(DataModel):
    updated_count: int


class EntityTextEmbedder(Component):
    """For every `:__Entity__` node, embed its longest string property → e.embedding."""

    def __init__(self, driver, label: str = '__Entity__', embedder=None):
        self.driver = driver
        self.label = label
        self.embedder = embedder

    async def run(self) -> EmbedEntityResult:
        updated = 0
        with self.driver.session(database=NEO4J_DATABASE) as session:
            result = session.run(
                f'MATCH (e:`{self.label}`) RETURN elementId(e) AS id, properties(e) AS props'
            )
            for record in result:
                node_id = record['id']
                props = record['props'] or {}
                str_props = {k: v for k, v in props.items() if isinstance(v, str) and v.strip()}
                if not str_props:
                    continue
                _, text = max(str_props.items(), key=lambda kv: len(kv[1]))
                emb = self.embedder.embed_query(text)
                session.run(
                    'MATCH (e) WHERE elementId(e) = $id SET e.embedding = $embedding',
                    id=node_id, embedding=emb,
                )
                updated += 1
        print(f'Embedded {updated} entity nodes.')
        return EmbedEntityResult(updated_count=updated)


entity_embedder = EntityTextEmbedder(driver=driver, embedder=query_embedder)
print('EntityTextEmbedder ready ✔')
# result = await entity_embedder.run()

EntityTextEmbedder ready ✔


In [16]:

from neo4j_graphrag.experimental.components.kg_writer import Neo4jWriter

eg_writer = Neo4jWriter(
    driver=driver,
    neo4j_database=NEO4J_DATABASE,
    batch_size=1000,
)
print('Neo4jWriter ready ✔')

Neo4jWriter ready ✔


In [17]:
from neo4j_graphrag.experimental.pipeline import Pipeline

pipe = Pipeline()

# 1. Register components
pipe.add_component(loader,              'data_loader')
pipe.add_component(splitter,            'text_splitter')
pipe.add_component(text_chunk_embedder, 'chunk_embedder')
pipe.add_component(schema_builder,      'schema')
pipe.add_component(entity_extractor,    'entity_extractor')
pipe.add_component(eg_writer,           'eg_writer')
pipe.add_component(resolver_title,      'resolver_title')
pipe.add_component(resolver_name,       'resolver_name')
pipe.add_component(entity_embedder,     'entity_embedder')

# 2. Wire data flow
pipe.connect('data_loader',    'text_splitter',    input_config={'page_data':   'data_loader'})
pipe.connect('text_splitter',  'chunk_embedder',   input_config={'text_chunks': 'text_splitter'})
pipe.connect('chunk_embedder', 'entity_extractor', input_config={'chunks':      'chunk_embedder'})
pipe.connect('schema',         'entity_extractor', input_config={'schema':      'schema'})
pipe.connect('entity_extractor','eg_writer',       input_config={'graph':       'entity_extractor'})

# Post-write components — they operate directly on the DB, so no input mapping is needed
pipe.connect('eg_writer', 'resolver_title',  {})
pipe.connect('eg_writer', 'resolver_name',   {})
pipe.connect('eg_writer', 'entity_embedder', {})

In [18]:
pipe_inputs = {
    'data_loader': {'path': DATA_PATH},
    'schema': {
        'node_types':         node_types,
        'relationship_types': relationship_types,
        'patterns':           patterns,
    },
}

In [ ]:
result = await pipe.run(pipe_inputs)
print('\nPipeline finished ✔')
print(result)

Initial DataFrame shape: (4869, 26)
Loaded 4869 movie rows from TMDB_IMDB_Movies_Dataset_filtered.xlsx (no filtering applied).
Built 4869 TextChunks (one per movie).


ERROR:neo4j_graphrag.experimental.components.entity_relation_extractor:LLM response has improper format for chunk_index=30
ERROR:neo4j_graphrag.experimental.components.entity_relation_extractor:LLM response has improper format for chunk_index=35
